# recs_010 - Two-Stage Eval (Habit -> Session)

Purpose: evaluate three families under the same contract:
- Family 1 (incumbent single-stage): `raw`, `popularity_train`, `multi_mean_train`
- Family 2 (single-stage fused): `q_eff = normalize(alpha*u_behavior + beta*u_reviews + gamma*q_session)`
- Family 3 (two-stage): retrieve top-M with habit vectors, rerank with `q_session`

This notebook is intentionally lean and artifact-driven.
Reference docs:
- `docs/eval_contract.md`
- `docs/recommender_transition_plan.md`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from steam_review_ml.recommender import evaluation as ev


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


# ---- top-level config ----
REPO_ROOT = _find_repo_root(Path.cwd())
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "recs"
EVAL_DIR = ARTIFACT_DIR / "eval"

SPLIT = "val"
K_FINAL = 10
M_STAGE1 = 100
K_PERSONALIZATION = 10
MAX_EXAMPLES = 12_500
RANDOM_SEED = 2026

ALPHA_BEHAVIOR = 0.45
BETA_REVIEWS = 0.45
GAMMA_SESSION = 0.10

METHODS_INCUMBENT = ["raw", "popularity_train", "multi_mean_train"]
SUPPORT_BUCKETS = ["0", "1", "2-3", "4-7", "8+"]
SLICE_RULES = {
    "slice_a_multi_target": "n_eval_targets >= 2",
    "slice_b_single_target": "n_eval_targets == 1",
    "slice_c_zero_target": "n_eval_targets == 0",
}

ENABLE_PLOTS = False

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 3: shared helpers (single definition source)

def l2_normalize(v: np.ndarray) -> np.ndarray:
    arr = np.asarray(v, dtype=np.float32).ravel()
    nrm = float(np.linalg.norm(arr))
    if nrm <= 1e-12:
        return arr
    return (arr / nrm).astype(np.float32)


def support_bucket(n: int) -> str:
    n = int(n)
    if n <= 0:
        return "0"
    if n == 1:
        return "1"
    if n <= 3:
        return "2-3"
    if n <= 7:
        return "4-7"
    return "8+"


def jaccard(a: set[int], b: set[int]) -> float:
    u = a | b
    if not u:
        return 1.0
    return len(a & b) / len(u)


def rank_rows(scores: np.ndarray) -> np.ndarray:
    return np.argsort(-scores)


def scores_from_query(q: np.ndarray, X: np.ndarray, app_to_row: dict[int, int], query_app_id: int) -> np.ndarray:
    s = (X @ q).astype(np.float32)
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return s


def eval_row(ranked: np.ndarray, positives: set[int], app_ids: np.ndarray, k: int) -> dict[str, float]:
    return {
        "Hit@K": ev.hit_rate_at_k(ranked, positives, k, app_ids),
        "Recall@K": ev.recall_at_k(ranked, positives, k, app_ids),
        "MAP@K": ev.average_precision_at_k(ranked, positives, k, app_ids),
        "NDCG@K": ev.ndcg_at_k(ranked, positives, k, app_ids),
        "MRR": ev.mrr(ranked, positives, app_ids),
    }


def slice_name_from_n_targets(n_eval_targets: int) -> str:
    if n_eval_targets >= 2:
        return "slice_a_multi_target"
    if n_eval_targets == 1:
        return "slice_b_single_target"
    return "slice_c_zero_target"

In [3]:
# Cell 4: Task A-compatible example/cohort construction
inputs = ev.prepare_eval_inputs(
    repo_root=REPO_ROOT,
    split=SPLIT,
    active_cohort="all",
    max_examples=MAX_EXAMPLES,
    support_app_filter_mode="strict",
    cohort_sizing={},
    min_review_chars=30,
    max_train_rows_per_user=5,
    random_seed=RANDOM_SEED,
    artifact_dir=ARTIFACT_DIR,
    verbose=True,
)

examples = inputs.examples
X = inputs.embedding_matrix
app_ids = inputs.app_ids
app_to_row = inputs.app_to_row
retriever = inputs.retriever

example_meta = pd.DataFrame(
    {
        "ex_idx": np.arange(len(examples), dtype=int),
        "user_id": [str(ex["user_id"]) for ex in examples],
        "query_app_id": [int(ex["query_app_id"]) for ex in examples],
        "n_eval_targets": [int(ex["n_eval_targets"]) for ex in examples],
        "n_support_train": [int(len(ex.get("support_texts_train", []))) for ex in examples],
    }
)
example_meta["slice_name"] = example_meta["n_eval_targets"].map(slice_name_from_n_targets)
example_meta["train_support_bucket"] = example_meta["n_support_train"].map(support_bucket)

print("examples:", len(examples))
display(example_meta.head())

Loaded split rows: eval=1,617,344 train=3,878,131 split_used=val


build eval examples: 100%|██████████| 12500/12500 [00:00<00:00, 83879.64row/s]


Prepared evaluation inputs: records=1,400,227 sampled=12,500 evaluable_examples=46
examples: 46


,ex_idx,user_id,query_app_id,n_eval_targets,n_support_train,slice_name,train_support_bucket
0,0,76561198001664105,582010,1,2,slice_b_single_target,2-3
1,1,76561198167070338,286160,1,5,slice_b_single_target,4-7
2,2,76561198056188104,524220,1,2,slice_b_single_target,2-3
3,3,76561198061610453,524220,1,1,slice_b_single_target,1
4,4,76561198001664105,292030,1,2,slice_b_single_target,2-3


In [4]:
# Cell 5: build vectors (u_behavior, u_reviews, q_session) with fallback rules
vector_rows = []
for ex_idx, ex in enumerate(examples):
    q_session = retriever.embed_text(str(ex["query_text"]))

    support_texts = [str(t).strip() for t in ex.get("support_texts_train", []) if str(t).strip()]
    if support_texts:
        support_vecs = np.stack([retriever.embed_text(t) for t in support_texts], axis=0).astype(np.float32)
        u_reviews = l2_normalize(support_vecs.mean(axis=0))
    else:
        u_reviews = q_session

    support_rows = ex.get("train_review_rows", [])
    support_app_ids = sorted({int(r["app_id"]) for r in support_rows if int(r["app_id"]) in app_to_row})
    if support_app_ids:
        emb = np.stack([X[app_to_row[a]] for a in support_app_ids], axis=0).astype(np.float32)
        u_behavior = l2_normalize(emb.mean(axis=0))
    else:
        u_behavior = u_reviews

    q_eff = l2_normalize(ALPHA_BEHAVIOR * u_behavior + BETA_REVIEWS * u_reviews + GAMMA_SESSION * q_session)
    habit_fused = l2_normalize(0.5 * u_behavior + 0.5 * u_reviews)

    vector_rows.append(
        {
            "ex_idx": ex_idx,
            "q_session": q_session,
            "u_reviews": u_reviews,
            "u_behavior": u_behavior,
            "habit_fused": habit_fused,
            "q_eff": q_eff,
        }
    )

vector_store = {int(r["ex_idx"]): r for r in vector_rows}
print("vectorized examples:", len(vector_store))

2026-05-07 12:11:02.331135: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778170262.341694  581955 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778170262.345395  581955 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778170262.398734  581955 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778170262.398759  581955 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778170262.398760  581955 computation_placer.cc:177] computation placer alr

vectorized examples: 46


In [5]:
# Cell 6: Stage 1 candidate retrieval eval (Recall@M, Hit@M)
stage1_method_vectors = {
    "stage1_u_behavior": "u_behavior",
    "stage1_u_reviews": "u_reviews",
    "stage1_habit_fused": "habit_fused",
}

stage1_rows = []
for ex_idx, ex in enumerate(examples):
    positives = set(int(a) for a in ex["positives"])
    if not positives:
        continue
    for method_name, vec_key in stage1_method_vectors.items():
        q = vector_store[ex_idx][vec_key]
        ranked = rank_rows(scores_from_query(q, X, app_to_row, int(ex["query_app_id"])))
        topM = ranked[:M_STAGE1]
        rec_m = ev.recall_at_k(topM, positives, M_STAGE1, app_ids)
        hit_m = ev.hit_rate_at_k(topM, positives, M_STAGE1, app_ids)
        stage1_rows.append(
            {
                "method": method_name,
                "ex_idx": ex_idx,
                "Recall@M": rec_m,
                "Hit@M": hit_m,
            }
        )

stage1_per_example = pd.DataFrame(stage1_rows)
stage1_table = (
    stage1_per_example.groupby("method", observed=True)[["Recall@M", "Hit@M"]]
    .mean()
    .reset_index()
    .sort_values(["Recall@M", "Hit@M"], ascending=False)
)

display(stage1_table)

,method,Recall@M,Hit@M
0,stage1_habit_fused,0.456522,0.456522
2,stage1_u_reviews,0.413043,0.413043
1,stage1_u_behavior,0.369565,0.369565


In [6]:
# Cell 7: Stage 2 rerank eval + incumbent baselines
rng = np.random.default_rng(RANDOM_SEED)
inc_registry = ev._build_method_registry(
    retriever=retriever,
    X=X,
    pop_row=inputs.pop_row,
    app_to_row=app_to_row,
    multi_max_reviews=5,
    rng=rng,
    mask_query_app=True,
)
inc_registry = {m: inc_registry[m] for m in METHODS_INCUMBENT}

rows = []
for ex_idx, ex in enumerate(examples):
    positives = set(int(a) for a in ex["positives"])
    if not positives:
        continue

    # Family 1 and 2 (single-stage)
    single_stage_scores = {
        **{m: inc_registry[m](ex) for m in METHODS_INCUMBENT},
        "single_fused_qeff": scores_from_query(vector_store[ex_idx]["q_eff"], X, app_to_row, int(ex["query_app_id"])),
    }
    for method_name, s in single_stage_scores.items():
        ranked = rank_rows(s)
        metric_vals = eval_row(ranked, positives, app_ids, K_FINAL)
        rows.append(
            {
                "method": method_name,
                "family": "incumbent" if method_name in METHODS_INCUMBENT else "single_fused",
                "ex_idx": ex_idx,
                **metric_vals,
            }
        )

    # Family 3 (two-stage: retrieve with habit, rerank with session)
    q_session = vector_store[ex_idx]["q_session"]
    two_stage_sources = {
        "two_stage_behavior_session": vector_store[ex_idx]["u_behavior"],
        "two_stage_reviews_session": vector_store[ex_idx]["u_reviews"],
        "two_stage_habit_fused_session": vector_store[ex_idx]["habit_fused"],
    }
    for method_name, q_stage1 in two_stage_sources.items():
        s1 = scores_from_query(q_stage1, X, app_to_row, int(ex["query_app_id"]))
        cands = rank_rows(s1)[:M_STAGE1]
        s2 = scores_from_query(q_session, X, app_to_row, int(ex["query_app_id"]))
        rerank_order = cands[np.argsort(-s2[cands])]
        metric_vals = eval_row(rerank_order, positives, app_ids, K_FINAL)
        rows.append(
            {
                "method": method_name,
                "family": "two_stage",
                "ex_idx": ex_idx,
                **metric_vals,
            }
        )

per_example_table = pd.DataFrame(rows).merge(example_meta, on="ex_idx", how="left")

overall_table = (
    per_example_table.groupby(["family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
    .sort_values(["NDCG@K", "MAP@K", "MRR"], ascending=False)
)

by_slice_table = (
    per_example_table.groupby(["slice_name", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

display(overall_table.head(20))

,family,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
1,incumbent,popularity_train,0.282609,0.282609,0.112707,0.151571,0.132568
4,two_stage,two_stage_behavior_session,0.108696,0.108696,0.029978,0.048080,0.039093
5,two_stage,two_stage_habit_fused_session,0.108696,0.108696,0.022214,0.041611,0.034642
0,incumbent,multi_mean_train,0.108696,0.108696,0.018565,0.038688,0.030217
3,single_fused,single_fused_qeff,0.065217,0.065217,0.009834,0.022236,0.025591
6,two_stage,two_stage_reviews_session,0.065217,0.065217,0.009446,0.021848,0.023832
2,incumbent,raw,0.065217,0.065217,0.008929,0.021351,0.021316


In [7]:
# Cell 8: support-bucket/popularity-decile cross-sections + deltas
by_support_table = (
    per_example_table.groupby(["train_support_bucket", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

app_pop = {int(a): float(c) for a, c in zip(app_ids, inputs.pop_row)}
pos_pop_rows = []
for ex_idx, ex in enumerate(examples):
    vals = [app_pop.get(int(a), 0.0) for a in ex["positives"]]
    pos_pop_rows.append({"ex_idx": ex_idx, "pos_pop_mean": float(np.mean(vals)) if vals else np.nan})
ex_pop = pd.DataFrame(pos_pop_rows)
valid = ex_pop["pos_pop_mean"].notna()
if valid.sum() > 0:
    ex_pop.loc[valid, "pos_pop_decile"] = pd.qcut(
        ex_pop.loc[valid, "pos_pop_mean"], q=10, labels=[f"D{i}" for i in range(1, 11)], duplicates="drop"
    )

by_pop_decile_table = (
    per_example_table.merge(ex_pop[["ex_idx", "pos_pop_decile"]], on="ex_idx", how="left")
    .dropna(subset=["pos_pop_decile"])
    .groupby(["pos_pop_decile", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

anchors = overall_table[overall_table["method"].isin(["raw", "popularity_train"])][["method", "Hit@K", "NDCG@K", "MRR"]]
anchor_map = {r["method"]: r for _, r in anchors.iterrows()}

deltas = []
for _, r in overall_table.iterrows():
    for anchor in ["raw", "popularity_train"]:
        if anchor not in anchor_map:
            continue
        a = anchor_map[anchor]
        deltas.append(
            {
                "method": r["method"],
                "family": r["family"],
                "anchor": anchor,
                "Hit@10_delta_vs_anchor": float(r["Hit@K"] - a["Hit@K"]),
                "NDCG@10_delta_vs_anchor": float(r["NDCG@K"] - a["NDCG@K"]),
                "MRR_delta_vs_anchor": float(r["MRR"] - a["MRR"]),
            }
        )

delta_vs_baselines_table = pd.DataFrame(deltas)

display(by_support_table.head(20))
display(by_pop_decile_table.head(20))
display(delta_vs_baselines_table.head(20))

,train_support_bucket,family,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
0,0,incumbent,multi_mean_train,0.125000,0.125000,0.017857,0.041667,0.026337
1,0,incumbent,popularity_train,0.125000,0.125000,0.125000,0.125000,0.152574
2,0,incumbent,raw,0.125000,0.125000,0.017857,0.041667,0.026337
3,0,single_fused,single_fused_qeff,0.125000,0.125000,0.017857,0.041667,0.026337
4,0,two_stage,two_stage_behavior_session,0.125000,0.125000,0.017857,0.041667,0.021645
5,0,two_stage,two_stage_habit_fused_session,0.125000,0.125000,0.017857,0.041667,0.021645
6,0,two_stage,two_stage_reviews_session,0.125000,0.125000,0.017857,0.041667,0.021645
7,1,incumbent,multi_mean_train,0.111111,0.111111,0.027778,0.047567,0.045295
8,1,incumbent,popularity_train,0.388889,0.388889,0.099537,0.167111,0.115825
9,1,incumbent,raw,0.111111,0.111111,0.014881,0.036044,0.031811


,pos_pop_decile,family,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
0,D1,incumbent,multi_mean_train,0.2,0.2,0.022222,0.060206,0.030086
1,D1,incumbent,popularity_train,0.0,0.0,0.000000,0.000000,0.006641
2,D1,incumbent,raw,0.0,0.0,0.000000,0.000000,0.014424
3,D1,single_fused,single_fused_qeff,0.0,0.0,0.000000,0.000000,0.029262
4,D1,two_stage,two_stage_behavior_session,0.0,0.0,0.000000,0.000000,0.025649
5,D1,two_stage,two_stage_habit_fused_session,0.0,0.0,0.000000,0.000000,0.020846
6,D1,two_stage,two_stage_reviews_session,0.0,0.0,0.000000,0.000000,0.019524
7,D2,incumbent,multi_mean_train,0.2,0.2,0.066667,0.100000,0.076232
8,D2,incumbent,popularity_train,0.0,0.0,0.000000,0.000000,0.011634
9,D2,incumbent,raw,0.0,0.0,0.000000,0.000000,0.014088


,method,family,anchor,Hit@10_delta_vs_anchor,NDCG@10_delta_vs_anchor,MRR_delta_vs_anchor
0,popularity_train,incumbent,raw,0.217391,0.130220,0.111252
1,popularity_train,incumbent,popularity_train,0.000000,0.000000,0.000000
2,two_stage_behavior_session,two_stage,raw,0.043478,0.026729,0.017777
3,two_stage_behavior_session,two_stage,popularity_train,-0.173913,-0.103491,-0.093475
4,two_stage_habit_fused_session,two_stage,raw,0.043478,0.020260,0.013326
5,two_stage_habit_fused_session,two_stage,popularity_train,-0.173913,-0.109960,-0.097926
6,multi_mean_train,incumbent,raw,0.043478,0.017337,0.008901
7,multi_mean_train,incumbent,popularity_train,-0.173913,-0.112883,-0.102351
8,single_fused_qeff,single_fused,raw,0.000000,0.000886,0.004275
9,single_fused_qeff,single_fused,popularity_train,-0.217391,-0.129335,-0.106977


In [8]:
# Cell 9: personalization diagnostics (non-gating)
all_methods = sorted(per_example_table["method"].unique().tolist())

# Build score fns for personalization diagnostics.
def _score_fn_for_method(method_name: str):
    if method_name in METHODS_INCUMBENT:
        return lambda ex, m=method_name: inc_registry[m](ex)

    if method_name == "single_fused_qeff":
        return lambda ex, _m=method_name: scores_from_query(
            vector_store[int(ex["_ex_idx"])]["q_eff"], X, app_to_row, int(ex["query_app_id"])
        )

    mapping = {
        "two_stage_behavior_session": "u_behavior",
        "two_stage_reviews_session": "u_reviews",
        "two_stage_habit_fused_session": "habit_fused",
    }
    if method_name in mapping:
        key = mapping[method_name]

        def _fn(ex, _k=key):
            ex_idx = int(ex["_ex_idx"])
            q_session = vector_store[ex_idx]["q_session"]
            q_stage1 = vector_store[ex_idx][_k]
            s1 = scores_from_query(q_stage1, X, app_to_row, int(ex["query_app_id"]))
            cands = rank_rows(s1)[:M_STAGE1]
            s2 = scores_from_query(q_session, X, app_to_row, int(ex["query_app_id"]))
            out = np.full_like(s2, -np.inf)
            out[cands] = s2[cands]
            return out

        return _fn

    raise KeyError(method_name)

examples_for_personalization = []
for ex_idx, ex in enumerate(examples):
    ex_copy = dict(ex)
    ex_copy["_ex_idx"] = ex_idx
    examples_for_personalization.append(ex_copy)

methods_for_personalization = {m: _score_fn_for_method(m) for m in all_methods}
personalization_table = ev._table_personalization(
    methods=methods_for_personalization,
    examples=examples_for_personalization,
    X=X,
    app_ids=app_ids,
    pop_row=inputs.pop_row,
    k_personalization=K_PERSONALIZATION,
    verbose=False,
)

display(personalization_table.sort_values("method"))

,method,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,multi_mean_train,0.154878,0.558730,9.813756,0.977476
1,popularity_train,0.212808,0.034921,5.325982,0.000000
2,raw,0.169158,0.650794,9.905556,0.972495
3,single_fused_qeff,0.132671,0.504762,9.483412,0.981312
4,two_stage_behavior_session,0.143034,0.574603,9.756479,0.972645
5,two_stage_habit_fused_session,0.147884,0.587302,9.911898,0.972368
6,two_stage_reviews_session,0.158045,0.600000,9.720842,0.978897


In [9]:
# Cell 10: compact summary + optional plots + artifact write
OUTPUT_PATH = ARTIFACT_DIR / "eval_two_stage_summary.csv"

summary_tables = {
    "stage1": stage1_table,
    "overall": overall_table,
    "by_slice": by_slice_table,
    "by_support": by_support_table,
    "by_pop_decile": by_pop_decile_table,
    "delta_vs_baselines": delta_vs_baselines_table,
    "personalization": personalization_table,
}

for name, table in summary_tables.items():
    print(f"{name:>18}: rows={len(table)}")

combined = []
for name, table in summary_tables.items():
    t = table.copy()
    t.insert(0, "section", name)
    combined.append(t)

summary_artifact = pd.concat(combined, ignore_index=True, sort=False)
summary_artifact.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote summary artifact: {OUTPUT_PATH}")

# Optional compact family-level summary
family_best = (
    overall_table.sort_values(["family", "NDCG@K", "MAP@K", "MRR"], ascending=[True, False, False, False])
    .groupby("family", as_index=False)
    .head(1)
)
display(family_best)

            stage1: rows=3
           overall: rows=7
          by_slice: rows=7
        by_support: rows=28
     by_pop_decile: rows=70
delta_vs_baselines: rows=14
   personalization: rows=7
Wrote summary artifact: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_two_stage_summary.csv


,family,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
1,incumbent,popularity_train,0.282609,0.282609,0.112707,0.151571,0.132568
3,single_fused,single_fused_qeff,0.065217,0.065217,0.009834,0.022236,0.025591
4,two_stage,two_stage_behavior_session,0.108696,0.108696,0.029978,0.048080,0.039093
